Imports and Libraries needed

In [1]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from langchain_ollama import ChatOllama

d:\Document\GitHub\p1\break\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\DELL\AppData\Local\Temp\ipykernel_15084\1851497939.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


Setting AI's API

In [2]:
model= "llama3.2"
db_name='vector_db'
load_dotenv(override=True)
openai_api_key=os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print("The api is working")
else:
    print("Key not set")

The api is working


Setting Knowledge Base / Vector DB

In [3]:
knowledge_base="knowledge_base/**/*.md"
files =glob.glob(knowledge_base, recursive=True)
print(f"{len(files)} files found in the Knowledge base")

all_knowledge_base = ""

for fp in files:
    with open(fp, 'r',encoding='utf-8') as f:
        all_knowledge_base += f.read()
        all_knowledge_base += "\n\n"

print(f"all {len(all_knowledge_base):,} characters are found in the knowledge base")

5 files found in the Knowledge base
all 12,558 characters are found in the knowledge base


taking all in the knowledge base then tokenize it, Count total tokens 

In [4]:
#encoding =tiktoken.encoding_for_model(model) #tokeniser corresponding to a specific model in the OpenAI API for PAID API
tiktoken.model.MODEL_TO_ENCODING[model] = "cl100k_base"
encoding = tiktoken.encoding_for_model(model)

tokens=encoding.encode(all_knowledge_base) #translating the text into numbers
token_count=len(tokens)
print(f"There are {token_count} tokens")

There are 2782 tokens


Loading all knowledge_base using langchain

In [5]:
folders=glob.glob("knowledge_base/*")

documents=[]
for f in folders:
    doc_type=os.path.basename(f)
    loader=DirectoryLoader(f, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs= loader.load()
    for doc in folder_docs:
        doc.metadata['doc_type'] = doc_type
        documents.append(doc) #getting all content of knowledge base and putting it all in one array
print(f'loaded {len(documents)} documents')

loaded 5 documents


Splitting document into chunks using RecursiveCharacterSplitter

In [6]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=250)
#text_splitter=RecursiveCharacterTextSplitter(chunk_size=1200)
chunks=text_splitter.split_documents(documents)

print(f'Divided into {len(chunks)} chunks')
print(f'first chunk: \n\n{chunks[0]}')  

Divided into 21 chunks
first chunk: 

page_content='# About RivanCyber

**Cisco • CompTIA • CCNA • CCNP • Fullstack** *Flexible Weekend & Evening Schedules | Industry-Standard Portfolio Projects*

RivanCyber Training Institute, Inc. is a premier IT training provider based in the Philippines. Established in the year 2000, the institute has built a twenty-six-year reputation for excellence in delivering hands-on, mentor-led IT education. 

We specialize in transforming ambitious beginners into certified professionals and helping seasoned IT specialists upscale their careers to meet global standards.

---

### 🎯 Mission
To provide "zero-to-hero" IT training that bridges the gap between theoretical knowledge and real-world application. We focus on job-ready skills, equipping students and professionals with the expertise needed to excel in networking, cybersecurity, and modern software development.' metadata={'source': 'knowledge_base\\company\\about.md', 'doc_type': 'company'}


if using llamaindex

In [7]:
'''

from llama_index.core.node_parser import SentenceSplitter

text_splitter = SentenceSplitter(
    chunk_size=1000, 
    chunk_overlap=200
)

chunks = text_splitter.get_nodes_from_documents(documents)

print(f'Divided into {len(chunks)} chunks')
print(f'first chunk: \n\n{chunks[0].text}')
'''

"\n\nfrom llama_index.core.node_parser import SentenceSplitter\n\ntext_splitter = SentenceSplitter(\n    chunk_size=1000, \n    chunk_overlap=200\n)\n\nchunks = text_splitter.get_nodes_from_documents(documents)\n\nprint(f'Divided into {len(chunks)} chunks')\nprint(f'first chunk: \n\n{chunks[0].text}')\n"

Turning text to vector

In [8]:
#from langchain_google_genai import GoogleGenerativeAIEmbeddings # for gemini api only
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key="AQ.Ab8RN6L4hCkz8j8AKmIU52LPY1kXBQNUUs80LyYxJWj3pJMWog"  ) #pip install langchain-google-genai
#if bayad
#embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11970.78it/s]


In [21]:
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection() #deletes all in old database
vectorstore=Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name) # putting the chunks in the chroma
print(f'vectorstore created with {vectorstore._collection.count()} documents')

vectorstore created with 21 documents


Check vector

In [22]:
collection=vectorstore._collection
count=collection.count()
sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0] # get existing records and return dictionary staring to grab index 0 
dimension = len(sample_embedding)
print(f'There are {count:,} vectors with {dimension:,} dimensions in the vector store')

There are 21 vectors with 384 dimensions in the vector store


Visualization

In [23]:
result= collection.get(include=['embeddings', 'documents','metadatas'])
vectors=np.array(result['embeddings'])
documents=result['documents']
metadatas=result['metadatas']
doc_types= [metadata['doc_type'] for metadata in metadatas]
colors =[['blue','green','red','orange'][['company','contacts','services'].index(t)] for t in doc_types]


In [32]:
#visualize using t-sne
#pip install --upgrade nbformat
tsne = TSNE(n_components=2, random_state=42, perplexity=5)
#perplexity tells the algorithm how many "nearest neighbors" to look at when trying to cluster your data points together
#n-components tells the algorithm how many dimensions you want your final output to have
#random_state is used to make your results reproducible. The t-SNE algorithm is a bit chaotic

reduced_vectors = tsne.fit_transform(vectors)

#scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

3D visualization

In [31]:
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

Setting retriever  

In [14]:
load_dotenv(override=True)

False

In [24]:
from langchain_openai import ChatOpenAI
import gradio as gr #pip install gradio
from langchain_core.messages import SystemMessage, HumanMessage
retriever = vectorstore.as_retriever() #literally this lang AHAHAHHAHA

Resettle the embedding and vectorstore var

In [26]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=db_name, embedding_function=embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2771.15it/s]


Setting LLM

In [25]:
#llm = ChatOpenAI(temperature=0, model_name=model) for PAID api
llm= ChatOllama(model=model,temperature=0)

Setting System Prompt

In [27]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company RivanCyber Institute Training.
You are chatting with a user about CyberRivan Institute Training.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

Setting-up func for retrieving and running the prompt through the llm invoke()

In [28]:
def answer_mo_dis(question: str, history):
    docs= retriever.invoke(question)
    context="\n\n".join(doc.page_content for doc in docs)
    sys_prompt= SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response= llm.invoke([SystemMessage(content=sys_prompt), HumanMessage(content=question)])
    return response.content


In [29]:
answer_mo_dis("What is Rivan", [])

'RivanCyber Institute Training, Inc., also known as Rivan, is a well-established IT training provider based in the Philippines. They specialize in delivering hands-on instruction across various technical domains, including Networking, Cybersecurity, Cloud, and Enterprise I.T.\n\nWith over 26 years of experience in the industry, Rivan has built a reputation for providing high-quality, job-ready training that bridges the gap between theoretical knowledge and real-world application. Their focus is on equipping students and professionals with practical skills to excel in their chosen fields.\n\nRivan offers flexible scheduling options, including weekend and evening classes, to accommodate different learning styles and schedules. They also have a strong portfolio of industry-standard projects to help learners gain practical experience and build their portfolios.\n\nSome of the key areas that Rivan specializes in include:\n\n*   Networking\n*   Cybersecurity\n*   Cloud Computing\n*   Enterpr

Launch to website

In [30]:
gr.ChatInterface(answer_mo_dis).launch(share=True)

* Running on local URL:  http://127.0.0.1:7861

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
